# Scheme 5 — Independent B6 and Other18 Flat Models

Scheme 5 trains two independent XGBoost flat selectors:

- **B family:** 6 `-B-bnb` algorithms.
- **Other family:** the remaining 18 algorithms.

Both use `strat_y` and `strat_y_K`. Original instances are split first; family-local ties are expanded only in training with divided weights. Held-out test rows remain one original instance. The notebook orchestrates the same Python pipeline used by the bash scripts rather than reimplementing training/evaluation logic.

## Cell 1 — Verify Scheme 5 algorithm partition
Expected: 6 B algorithms, 18 Other algorithms, disjoint union = all 24.

In [ ]:
from restructured_pipeline.src.features import ALL_ALGOS, B_ALGOS, OTHER_ALGOS
assert len(B_ALGOS)==6
assert len(OTHER_ALGOS)==18
assert set(B_ALGOS).isdisjoint(OTHER_ALGOS)
assert set(B_ALGOS)|set(OTHER_ALGOS)==set(ALL_ALGOS)
print("B6:", B_ALGOS)
print("Other18:", OTHER_ALGOS)

## Cell 2 — Train all four Scheme 5 models
Runs B/Other × `strat_y`/`strat_y_K`. Each training run writes its model, entire-data predictions, wrong cases, metrics, confusion matrix, and split indices under `scheme5/<family>/xgboost`.

In [ ]:
import os, subprocess
os.environ["RUN_TAG"]="17sep2026"
subprocess.run(["bash","train_scheme5_xgb.sh"],check=True)

## Cell 3 — Predict, evaluate, and build tolerance reports
For every trained model, generates **held-out test** and **entire-data** predictions, evaluates the saved prediction CSV, and then builds 0/10/20/50% tolerance workbooks from existing predictions (no retraining).

In [ ]:
os.environ["TRAIN_TAG"]="17sep2026"
os.environ["ANALYSIS_TAG"]="17sep2026"
subprocess.run(["bash","run_scheme5_predict_evaluate_tolerance.sh"],check=True)

## Cell 4 — Failure analysis and mixed Top-K
Runs the existing compare → tolerance → important-failure pipeline separately for B and Other, enriches canonical `tol_0` important failures with Top-1…Top-6 information, then generates all B1–B3 × O1–O3 mixed combinations for both split modes. If a dataset/K is an important failure in both families, it remains **two family-specific rows**.

In [ ]:
os.environ["TRAIN_TAG"]="17sep2026"
os.environ["PREDICTION_TAG"]="17sep2026"
subprocess.run(["bash","run_scheme5_failure_analysis.sh"],check=True)

## Cell 5 — Inspect the final summary tables
Loads the two family failure summaries and both mixed Top-K summaries for quick validation/slide preparation.

In [ ]:
from pathlib import Path
import pandas as pd
TAG="17sep2026_17sep2026"
R=Path("restructured_pipeline/outputs/reports")
for family in ("b","other"):
    p=R/f"failure_analysis_scheme5_{family}_{TAG}.xlsx"
    print("\n",p)
    display(pd.read_excel(p,sheet_name="summary"))
for split in ("strat_y","strat_y_K"):
    p=R/f"scheme5_mixed_topk_{split}_{TAG}.xlsx"
    print("\n",p)
    display(pd.read_excel(p,sheet_name="summary"))

## Expected output layout

- `outputs/models/scheme5/b/xgboost` and `.../other/xgboost`
- `outputs/predictions/scheme5/b/xgboost` and `.../other/xgboost`
- `outputs/wrong_cases/scheme5/b/xgboost` and `.../other/xgboost`
- `outputs/metrics/scheme5/b/xgboost` and `.../other/xgboost`
- `outputs/reports/test_data/scheme5/<family>`
- `outputs/reports/entire_data/scheme5/<family>`
- `outputs/reports/failure_analysis_scheme5_<family>_<tags>.xlsx`
- `outputs/reports/scheme5_mixed_topk_<split>_<tags>.xlsx`

The filenames deliberately follow the Scheme 4 naming convention with `scheme5_b` / `scheme5_other`, so prediction discovery, evaluation, tolerance reporting, and failure analysis all resolve the same artifacts.